In [1]:
economy_urls = {
    "Defense_Budget_USD": "https://www.globalfirepower.com/defense-spending-budget.php",
    "External_Debt_USD": "https://www.globalfirepower.com/external-debt-by-country.php",
    "Purchasing_Power_Parity_USD": "https://www.globalfirepower.com/purchasing-power-parity.php",
    "Foreign_Exchange_Gold_Reserves_USD": "https://www.globalfirepower.com/reserves-of-foreign-exchange-and-gold.php"
}

In [2]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

In [4]:
def scrape_metric(url, metric_name):

    print(f"Scraping {metric_name}...")

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Failed to load {url}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    records = soup.find_all("div", class_="recordsetContainer")

    data = []

    for record in records:

        long_name = record.find("div", class_="longFormName")
        short_name = record.find("div", class_="shortFormName")
        value = record.find("div", class_="valueContainer")

        if value:

            if long_name:
                country = long_name.get_text(strip=True)
            elif short_name:
                country = short_name.get_text(strip=True)
            else:
                continue

            value = value.get_text(strip=True)

            data.append([country, value])

    df = pd.DataFrame(data, columns=["Country", metric_name])

    return df

In [5]:
dfs = []

for metric, url in economy_urls.items():

    df = scrape_metric(url, metric)

    if df is not None:
        dfs.append(df)

    time.sleep(2)

Scraping Defense_Budget_USD...
Scraping External_Debt_USD...
Scraping Purchasing_Power_Parity_USD...
Scraping Foreign_Exchange_Gold_Reserves_USD...


In [6]:
economy_df = dfs[0]

for df in dfs[1:]:

    economy_df = economy_df.merge(
        df,
        on="Country",
        how="outer"
    )

In [7]:
for col in economy_df.columns[1:]:

    economy_df[col] = (
        economy_df[col]
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
    )

    economy_df[col] = pd.to_numeric(
        economy_df[col],
        errors="coerce"
    )

In [8]:
print(economy_df.head())

       Country  Defense_Budget_USD  External_Debt_USD  \
0  Afghanistan           145000000         1929750000   
1      Albania           720037190         5363000000   
2      Algeria         25000000000         4764000000   
3       Angola         31200000000        45299000000   
4    Argentina           993919790        74362000000   

   Purchasing_Power_Parity_USD  Foreign_Exchange_Gold_Reserves_USD  
0                  82238000000                          8852092000  
1                  51360000000                          6516000000  
2                 722912000000                         83007000000  
3                 278239000000                         14243000000  
4                1213000000000                         29560000000  


In [9]:
economy_df.to_csv("economy.csv", index=False)

print("Economy dataset saved successfully!")

Economy dataset saved successfully!


In [10]:
print("Shape:", economy_df.shape)
print()

print(economy_df.info())
print()

print(economy_df.head(10))

Shape: (145, 5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 5 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   Country                             145 non-null    object
 1   Defense_Budget_USD                  145 non-null    int64 
 2   External_Debt_USD                   145 non-null    int64 
 3   Purchasing_Power_Parity_USD         145 non-null    int64 
 4   Foreign_Exchange_Gold_Reserves_USD  145 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 5.8+ KB
None

       Country  Defense_Budget_USD  External_Debt_USD  \
0  Afghanistan           145000000         1929750000   
1      Albania           720037190         5363000000   
2      Algeria         25000000000         4764000000   
3       Angola         31200000000        45299000000   
4    Argentina           993919790        74362000000   
5      Armenia          148000000